In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import re
import pandas as pd
from histcite.process_file import ProcessFile
from histcite.compute_metrics import ComputeMetrics
from histcite.network_graph import GraphViz

In [3]:
# 读取文件，去重，识别引文关系
folder_path = 'dataset/wos'
source_type = 'wos'

# folder_path = 'dataset/cssci'
# source_type = 'cssci'

# folder_path = 'dataset/scopus'
# source_type = 'scopus'

process = ProcessFile(folder_path,source_type)
process.concat_table() # 合并多个文件
process.process_citation() # 识别引用关系
docs_table = process.docs_table
reference_table = process.reference_table
docs_table.head()

共读取 2000 条数据，去重后剩余 2000 条


,doc_index,AU,first_AU,TI,SO,DT,DE,C3,CR,NR,...,J9,PY,VL,BP,DI,UT,reference,citation,LCR,LCS
0,0,"GLAUERT, JRW; KENNAWAY, JR; SLEEP, MR",GLAUERT JRW,DACTL - AN EXPERIMENTAL GRAPH REWRITING LANGUAGE,LECTURE NOTES IN COMPUTER SCIENCE,Article,DACTL; GRAPH REWRITING; COMPUTATIONAL MODEL; P...,University of East Anglia,"BAETEN JCM, 1987, LNCS, V257, P83; BARENDREGT ...",36,...,LECT NOTES COMPUT SC,1991,532,378,<NA>,WOS:A1991LE71500041,None,None,0,0
1,1,"PIELA, PC; EPPERLY, TG; WESTERBERG, KM; WESTER...",PIELA PC,ASCEND - AN OBJECT-ORIENTED COMPUTER ENVIRONME...,COMPUTERS & CHEMICAL ENGINEERING,Article,<NA>,Carnegie Mellon University,"Borning Alan, 1979, THESIS STANFORD U; EPPERLY...",19,...,COMPUT CHEM ENG,1991,15,53,10.1016/0098-1354(91)87006-U,WOS:A1991FA96900006,None,65;78,0,2
2,2,"TEETERS, JL",TEETERS JL,MDL - A SYSTEM FOR FAST SIMULATION OF LARGE LA...,SIMULATION,Article,MODEL DEVELOPMENT LANGUAGE; SIMULATION LANGUAG...,University of Southern California,"Carey R, 1975, THESIS U MASSACHUSET; DESCHUTTE...",14,...,SIMULATION,1991,56,369,10.1177/003754979105600606,WOS:A1991FV44300002,None,None,0,0
3,3,"MCBRIEN, P; NIEZETTE, M; PANTAZIS, D; SELTVEIT...",MCBRIEN P,A RULE LANGUAGE TO CAPTURE AND MODEL BUSINESS ...,LECTURE NOTES IN COMPUTER SCIENCE,Article,<NA>,University of Liege; University of Manchester,"BERZTISS A, 1986, INFORMATION SYSTEM D; BORGID...",23,...,LECT NOTES COMPUT SC,1991,498,307,<NA>,WOS:A1991FW90200015,None,None,0,0
4,4,"Kottemann, JE; Dolk, DR",Kottemann JE,Model Integration and Modeling Languages: A Pr...,INFORMATION SYSTEMS RESEARCH,Article,Model management; Model integration; Modeling ...,University of Michigan System; University of M...,"Blanning R. W., 1986, Decision Support Systems...",25,...,INFORM SYST RES,1992,3,1,10.1287/isre.3.1.1,WOS:000209837400001,None,None,0,0


In [4]:
reference_table.head()

,first_AU,PY,J9,VL,BP,DI,doc_index
0,BAETEN JCM,1987,LNCS,257,83,None,0
1,BARENDREGT HP,1987,LECT NOTES COMPUT SC,259,141,None,0
2,BARENDREGT HP,1989,PARALLEL COMPUT,9,163,10.1016/0167-8191(89)90126-9,0
3,DERSHOWITZ N,1989,HDB THEORETICAL CO B,<NA>,<NA>,None,0
4,EHRIG H,1979,LNCS,73,83,None,0


In [5]:
# 基本描述统计
cm = ComputeMetrics(docs_table, reference_table, source_type)
cm.write2excel(os.path.join(folder_path,'result','statistics.xlsx'))

图文件可以使用 [Graphviz在线编辑器](http://magjac.com/graphviz-visual-editor/) 或本地的 [Graphviz工具](https://graphviz.org/) 生成引文网络图。  

In [6]:
# 生成引文网络图文件

# 选取LSC最高的100篇文献
doc_indices = docs_table.sort_values('LCS', ascending=False).index[:100]
# 选取LSC大于5的文献
# doc_indices = docs_table[citation_table['LCS']>5].index

graph = GraphViz(docs_table, source_type)
graph_dot_file = graph.generate_dot_file(doc_indices,allow_external_node=False)

# 保存graph.dot文件
with open(os.path.join(folder_path,'result','graph.dot'),'w') as f:
    f.write(graph_dot_file)
print(graph_dot_file)

digraph metadata{
	rankdir = BT;
	{rank=same; 1997 31};
	{rank=same; 1999 58 68};
	{rank=same; 2000 77 82 86 97};
	{rank=same; 2002 120};
	{rank=same; 2003 135 143 144 149};
	{rank=same; 2004 161 165};
	{rank=same; 2005 193};
	{rank=same; 2006 219 236 240 245 253 254};
	{rank=same; 2007 265 272 273 275 287 288 297 301};
	{rank=same; 2008 317 325};
	{rank=same; 2009 381 389 391 410};
	{rank=same; 2011 492 501 502};
	{rank=same; 2012 541 576};
	{rank=same; 2013 587 597 609 623};
	{rank=same; 2014 644 645 662 670 708};
	{rank=same; 2015 739 748 749 793};
	{rank=same; 2016 821 827 859 875 913};
	{rank=same; 2017 915 926 975 992};
	{rank=same; 2018 1031};
	{rank=same; 2019 1174 1177 1178};
	1997 [ shape="plaintext" ];
	1999 [ shape="plaintext" ];
	2000 [ shape="plaintext" ];
	2002 [ shape="plaintext" ];
	2003 [ shape="plaintext" ];
	2004 [ shape="plaintext" ];
	2005 [ shape="plaintext" ];
	2006 [ shape="plaintext" ];
	2007 [ shape="plaintext" ];
	2008 [ shape="plaintext" ];
	2009 [ shape="p

In [7]:
# 导出图节点文件
graph_node_file = graph.generate_graph_node_file()
graph_node_file.to_excel(os.path.join(folder_path,'result','graph.node.xlsx'),index=False)
graph_node_file

,doc_index,AU,PY,SO,VL,BP,LCS,GCS
31,31,"Iyer, R; Ostendorf, M; Gish, H",1997,IEEE SIGNAL PROCESSING LETTERS,4,221,5,24
58,58,"Iyer, RM; Ostendorf, M",1999,IEEE TRANSACTIONS ON SPEECH AND AUDIO PROCESSING,7,30,10,55
68,68,"Iver, R; Ostendorf, M",1999,COMPUTER SPEECH AND LANGUAGE,13,267,7,14
77,77,"Bellegarda, JR",2000,IEEE TRANSACTIONS ON SPEECH AND AUDIO PROCESSING,8,76,7,43
82,82,"Bellegarda, JR",2000,PROCEEDINGS OF THE IEEE,88,1279,15,210
...,...,...,...,...,...,...,...,...
992,992,"Hellendoorn, VJ; Devanbu, P",2017,ESEC/FSE 2017: PROCEEDINGS OF THE 2017 11TH JO...,<NA>,763,4,135
1031,1031,"Kipyatkova, I",2018,SPEECH AND COMPUTER (SPECOM 2018),11096,291,2,3
1174,1174,"Alferez, M; Acher, M; Galindo, JA; Baudry, B; ...",2019,SOFTWARE QUALITY JOURNAL,27,307,2,12
1177,1177,"Huang, J; Zhou, WG; Li, HQ; Li, WP",2019,IEEE TRANSACTIONS ON CIRCUITS AND SYSTEMS FOR ...,29,2822,5,69
